<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/5%20-%20Exemplos%20de%20Aplica%C3%A7%C3%A3o/5.5_expectativa_de_vida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 5.5 — Expectativa de vida: regressão com dados de saúde dos países

Neste exemplo, a rede faz **regressão**: prevê a expectativa de vida (em anos) de um país num certo ano, a partir de indicadores de saúde e economia, como mortalidade adulta e infantil, vacinação, PIB, escolaridade e prevalência de HIV.

O código segue o mesmo esqueleto do notebook 4.6 (MNIST), com quatro diferenças:

| | 4.6 — MNIST | 5.5 — Expectativa de vida |
|---|---|---|
| entrada | imagem 28×28 achatada (784 pixels) | 20 indicadores numéricos de uma tabela |
| saída | 10 log-probabilidades (classificação) | 1 número, sem ativação (regressão) |
| função de custo | `F.nll_loss` | `nn.MSELoss` (erro quadrático médio) |
| avaliação | acurácia | erro médio em anos (MAE e RMSE) |

Nos exemplos de aplicação, a função de custo é sempre criada como um objeto do `torch.nn` e passada para `train` e `test` como `criterion`: `nn.MSELoss()` para regressão e `nn.CrossEntropyLoss()` para classificação. A `nn.CrossEntropyLoss` recebe os logits e aplica o log-softmax por dentro; equivale ao par `log_softmax` + `F.nll_loss` do 4.6, mas a rede termina direto na camada linear. O otimizador é o **SGD com momentum**.

Os dados reais também dão mais trabalho que o MNIST: há valores faltando, as colunas estão em escalas muito diferentes, e é preciso pensar em como separar treino e teste.

## Objetivos

- Preparar uma tabela real para a rede: tratar valores faltantes, padronizar as features e o alvo.
- Separar treino e teste **por país**, para medir o desempenho em países que a rede nunca viu.
- Adaptar o pipeline do 4.6 para regressão: uma saída sem ativação e `nn.MSELoss`.
- Interpretar o erro em anos e comparar a rede com um baseline simples.

## Importação das bibliotecas

In [ ]:
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

torch.manual_seed(1111)   # reprodutibilidade: pesos iniciais, dropout e embaralhamento
np.random.seed(1111)

device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

## Dados

Usamos o conjunto [Life Expectancy (WHO)](https://www.kaggle.com/datasets/kumarajarshi/life-expectancy-who), do Kaggle, montado com dados da Organização Mundial da Saúde e da ONU: 193 países, de 2000 a 2015, uma linha por país e ano. O endereço da API do Kaggle baixa o zip sem login. Se falhar, baixe pela página e coloque `Life Expectancy Data.csv` em `data/`.

In [ ]:
os.makedirs("data", exist_ok=True)
url = "https://www.kaggle.com/api/v1/datasets/download/kumarajarshi/life-expectancy-who"
zip_path = "data/life-expectancy-who.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("data")

df = pd.read_csv("data/Life Expectancy Data.csv")
print(df.shape)
df.head()

Alguns nomes de coluna têm espaços sobrando (`'Life expectancy '`, `' BMI '`). Tiramos esses espaços para não errar ao digitar os nomes.

In [ ]:
df.columns = df.columns.str.strip().str.replace('  ', ' ')
print(df.columns.tolist())

### Valores faltantes

`isna().sum()` conta, em cada coluna, quantas linhas não têm valor (`NaN`). A rede não aceita `NaN`: uma única entrada faltando contamina a saída e, depois, todos os gradientes.

In [ ]:
df.isna().sum().sort_values(ascending=False)

Decisões:

- **Alvo** (`Life expectancy`): as 10 linhas sem valor são descartadas; não há o que aprender com elas.
- **`Country`**: fica de fora das features. O nome do país não é uma medida, e queremos que a rede funcione para países que ela não viu.
- **`Status`** (`Developed`/`Developing`) é texto: vira 1 para país desenvolvido e 0 para país em desenvolvimento.
- **Features com valores faltantes**: preenchemos com a **mediana do treino** (mais adiante, depois de separar treino e teste).

In [ ]:
df = df.dropna(subset=['Life expectancy'])
df['Status'] = (df['Status'] == 'Developed').astype(float)

target = 'Life expectancy'
features = [c for c in df.columns if c not in ('Country', target)]
print(len(features), 'features:', features)
print(df[target].describe())

### Treino e teste por país

Cada país aparece 16 vezes (um por ano), e os indicadores de um país mudam pouco de um ano para o outro. Se sortearmos as **linhas**, o Brasil de 2010 pode cair no treino e o Brasil de 2011 no teste: a rede acerta o teste quase decorando o país, e o erro medido sai otimista.

Por isso sorteamos **países**: `GroupShuffleSplit` coloca todas as linhas de um mesmo grupo (aqui, o país) do mesmo lado. Os 20% de países do teste nunca são vistos no treino. O exercício 1 compara com o sorteio por linhas.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df['Country']))
df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]

print('treino:', len(df_train), 'linhas,', df_train['Country'].nunique(), 'países')
print('teste: ', len(df_test), 'linhas,', df_test['Country'].nunique(), 'países')
print('países em comum:', len(set(df_train['Country']) & set(df_test['Country'])))

### Valores faltantes e padronização

Tudo o que é calculado a partir dos dados (a mediana para preencher faltantes, a média e o desvio-padrão da padronização) usa **só o treino**. O teste faz o papel de dados novos: na hora de usar o modelo, não conheceríamos esses dados de antemão.

- `fillna(mediana)`: troca cada `NaN` pela mediana da coluna no treino.
- `StandardScaler`: padroniza cada feature para média 0 e desvio 1, como nos notebooks anteriores. As colunas têm escalas muito diferentes (população na casa dos milhões, IMC na casa das dezenas).
- O alvo também é padronizado (`y_scaler`). Com média 0 e desvio 1, o custo começa perto de 1 e a rede não precisa aprender a "chegar a 70". Para ler as previsões em anos, desfazemos a padronização com `y_scaler.inverse_transform`.

In [ ]:
medianas = df_train[features].median()
X_train = df_train[features].fillna(medianas).values
X_test = df_test[features].fillna(medianas).values
y_train = df_train[[target]].values      # [[target]]: shape (M, 1), uma coluna
y_test = df_test[[target]].values

X_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)
X_train_n, X_test_n = X_scaler.transform(X_train), X_scaler.transform(X_test)
y_train_n, y_test_n = y_scaler.transform(y_train), y_scaler.transform(y_test)

print('X_train:', X_train_n.shape, '| y_train:', y_train_n.shape)
print(f'alvo: média {y_scaler.mean_[0]:.1f} anos, desvio {y_scaler.scale_[0]:.1f} anos')

### Dataset e DataLoader

No 4.6, `datasets.MNIST` já entregava um `Dataset` pronto. Aqui os dados são tabelas, e `TensorDataset` resolve: ele recebe os tensores de entrada e de alvo, e `dataset[i]` devolve o par `(x_i, y_i)`. O `DataLoader` funciona igual ao do 4.6: mini-batches de 32 linhas no treino, embaralhados a cada época.

In [ ]:
train_dataset = TensorDataset(torch.tensor(X_train_n, dtype=torch.float32), torch.tensor(y_train_n, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test_n, dtype=torch.float32), torch.tensor(y_test_n, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

data, target_batch = next(iter(train_loader))
print('mini-batches por época:', len(train_loader))
print('mini-batch: data', tuple(data.shape), ' target', tuple(target_batch.shape))

## Modelo

A mesma estrutura da rede do 4.6, menor, porque a entrada tem 20 números em vez de 784:

- `fc1` (20 → 64) e `fc2` (64 → 32): camadas escondidas com ReLU e dropout;
- `fc3` (32 → 1): **uma** saída e **sem ativação** (nem softmax, nem ReLU), porque a expectativa de vida padronizada pode ser qualquer número real, inclusive negativo.

Não há `torch.flatten`: cada amostra já é um vetor.

In [ ]:
n_features = X_train_n.shape[1]

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)

    def forward(self, x):          # x: (N, 20)
        x = self.fc1(x)            # (N, 64)
        x = F.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)            # (N, 32)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc3(x)            # (N, 1): expectativa de vida padronizada
        return x

model = Net()
print(model)
print('parâmetros:', sum(p.numel() for p in model.parameters()))

## Treinamento

### Funções de treino e de teste

São as funções do 4.6 com duas trocas: o custo é `nn.MSELoss` (média de $(\hat{y} - y)^2$ no mini-batch), passado às funções como `criterion`, e não há acurácia. Para não imprimir dezenas de linhas por época, `train` devolve o custo médio da época, e o laço imprime uma linha por época.

In [ ]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()                       # modo de treino: dropout ativo
    total_loss = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(data)       # custo do mini-batch × nº de amostras
    return total_loss / len(train_loader.dataset)  # custo médio da época

In [ ]:
def test(model, device, test_loader, criterion):
    model.eval()                        # modo de avaliação: dropout desligado
    test_loss = 0
    with torch.no_grad():               # sem gradientes na avaliação
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * len(data)   # custo médio do lote × nº de amostras
    return test_loss / len(test_loader.dataset)

### Função de custo, otimizador e agendador

- **`nn.MSELoss()`**: o erro quadrático médio, criado uma vez e passado para `train` e `test`.
- **SGD com momentum** (`lr=0.01, momentum=0.9`): o gradiente descendente estocástico dos notebooks anteriores. O *momentum* faz cada passo levar em conta a direção dos passos anteriores, como uma bola descendo a ladeira: o treino oscila menos e anda mais rápido nas direções em que o gradiente é consistente.
- **StepLR** com `step_size=20, gamma=0.5`: a cada 20 épocas, a taxa de aprendizado cai pela metade.

São 60 épocas. Com 2.300 linhas, cada época leva uma fração de segundo.

In [ ]:
model = Net().to(device)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)

n_epochs = 60
train_losses, test_losses = [], []
for epoch in range(1, n_epochs + 1):
    train_loss = train(model, device, train_loader, optimizer, criterion)
    test_loss = test(model, device, test_loader, criterion)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Época {epoch:2d} | lr = {scheduler.get_last_lr()[0]:.5f} | custo treino: {train_loss:.4f} | custo teste: {test_loss:.4f}')
    scheduler.step()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, n_epochs + 1), train_losses, label='treino')
plt.plot(range(1, n_epochs + 1), test_losses, label='teste')
plt.xlabel('épocas'); plt.ylabel('custo (MSE no alvo padronizado)'); plt.yscale('log')
plt.legend(); plt.grid(True)
plt.show()

O custo de treino é medido com o dropout ligado e enquanto os pesos mudam; o de teste, com o dropout desligado e no fim da época. A distância entre as duas curvas mostra quanto pior é prever países novos do que países vistos no treino.

## Gravação e leitura do modelo

Como no 4.6: `state_dict` guarda os pesos, e `load_state_dict` os carrega numa rede com a mesma arquitetura. Para usar o modelo depois, é preciso guardar também as medianas e os dois scalers: sem eles não dá para preparar uma entrada nova nem ler a saída em anos.

In [ ]:
torch.save(model.state_dict(), "expectativa_vida_mlp.pt")

model2 = Net().to(device)
model2.load_state_dict(torch.load("expectativa_vida_mlp.pt", map_location=device))
print(f'custo no teste do modelo carregado: {test(model2, device, test_loader, criterion):.4f}')

## Avaliação

O custo está em unidades padronizadas. Para interpretá-lo, calculamos as previsões do conjunto de teste, desfazemos a padronização e medimos o erro em **anos**:

$$\text{MAE} = \frac{1}{M}\sum_{i=1}^{M} |\hat{y}_i - y_i| \qquad \text{RMSE} = \sqrt{\frac{1}{M}\sum_{i=1}^{M} (\hat{y}_i - y_i)^2}$$

em que $M$ é o número de linhas de teste, $y_i$ a expectativa de vida real e $\hat{y}_i$ a prevista. Para saber se o número é bom, comparamos com um **baseline** que não usa nenhuma feature: prever, para todo país, a média do treino.

In [ ]:
model.eval()
with torch.no_grad():
    pred_n = model(torch.tensor(X_test_n, dtype=torch.float32).to(device)).cpu().numpy()
pred = y_scaler.inverse_transform(pred_n)      # de volta para anos

mae = np.abs(pred - y_test).mean()
rmse = np.sqrt(((pred - y_test) ** 2).mean())
mae_base = np.abs(y_train.mean() - y_test).mean()
rmse_base = np.sqrt(((y_train.mean() - y_test) ** 2).mean())
print(f'rede:     MAE = {mae:.2f} anos | RMSE = {rmse:.2f} anos')
print(f'baseline: MAE = {mae_base:.2f} anos | RMSE = {rmse_base:.2f} anos')

In [ ]:
plt.figure(figsize=(5.5, 5.5))
plt.scatter(y_test, pred, s=8, alpha=0.5)
lims = [y_test.min() - 2, y_test.max() + 2]
plt.plot(lims, lims, 'r--', label='previsão perfeita')
plt.xlabel('expectativa de vida real (anos)'); plt.ylabel('prevista (anos)')
plt.title('Conjunto de teste (países não vistos)')
plt.legend(); plt.grid(True)
plt.show()

Cada país do teste tem uma série de 16 anos. Comparar a série real com a prevista mostra se a rede acompanha a tendência de cada país, além de acertar o nível médio.

In [ ]:
df_pred = df_test[['Country', 'Year', target]].copy()
df_pred['previsto'] = pred.ravel()
paises = df_pred.groupby('Country')[target].mean().sort_values().index[[0, len(set(df_pred['Country'])) // 2, -1]]

fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, pais in zip(axs, paises):
    d = df_pred[df_pred['Country'] == pais].sort_values('Year')
    ax.plot(d['Year'], d[target], 'o-', label='real')
    ax.plot(d['Year'], d['previsto'], 's--', label='prevista')
    ax.set_title(pais); ax.set_xlabel('ano'); ax.set_ylabel('anos'); ax.legend()
plt.tight_layout()
plt.show()

A rede prevê cada ano separadamente, sem saber que as linhas formam uma série; por isso a curva prevista oscila de um ano para o outro, sobretudo quando algum indicador do país muda bruscamente ou tem valor faltando (preenchido pela mediana). Mesmo assim, ela acompanha o nível e a tendência de países muito diferentes.

## Exercícios

1. Refaça a divisão sorteando **linhas** em vez de países (`train_test_split(df, test_size=0.2, random_state=42)`), repita o preenchimento, a padronização e o treino, e calcule o MAE no teste. Antes de executar, preveja se o MAE vai subir ou cair. Qual das duas medidas descreve melhor o erro que teríamos num país novo?

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: escreva sua solução aqui
print(f'MAE com divisão por país:  {mae:.2f} anos')
print(f'MAE com divisão por linha: {mae_linhas:.2f} anos')

*Escreva sua resposta aqui.*

2. Quais indicadores a rede mais usa? Uma forma simples de medir é a **importância por permutação**: embaralhe só uma coluna de `X_test_n` (as outras ficam como estão), calcule o MAE de novo e veja quanto ele piorou. Quanto maior a piora, mais a rede depende daquela feature. Faça isso para todas as features e mostre as 5 mais importantes. O resultado faz sentido?

In [ ]:
# TODO: escreva sua solução aqui
print(importancia.head(5).round(2))

*Escreva sua resposta aqui.*

## Aprendizados

- Dados tabulares reais pedem preparação antes da rede: tratar valores faltantes, converter texto em números e padronizar features e alvo, sempre com estatísticas calculadas só no treino.
- A forma de separar o teste deve imitar o uso real. Com várias linhas por país, sortear linhas deixa o teste fácil demais; sortear países mede o erro em países novos.
- Para regressão, a rede termina numa saída sem ativação e usa `nn.MSELoss`. O resto do pipeline do 4.6 (DataLoader, `train`/`test`, SGD, StepLR, `state_dict`) não muda.
- O erro só tem significado em unidades do problema (anos) e comparado com um baseline.